[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/01_alumno_exploracion.ipynb)

# MLY1101 · Machine Learning — Semana 01
## EA1 · Análisis y Preprocesamiento de Datos

**Resultado de aprendizaje (RA1):** implementar estrategias y técnicas de preprocesamiento
en el diseño de soluciones de Machine Learning, con un tratamiento responsable de la información.

---

### La idea central de hoy

Un proyecto de Machine Learning **no empieza eligiendo un algoritmo**. Empieza entendiendo
el problema y mirando los datos:

```
Problema → Datos → Exploración → Preprocesamiento → Modelamiento → Evaluación → Interpretación
           └────────── aquí estamos hoy ──────────┘
```

Hoy no vamos a entrenar ningún modelo. Vamos a hacer algo que decide el éxito o el fracaso
del modelo que entrenaremos más adelante: **entender y limpiar los datos**.

> Un modelo entrenado con datos que nadie revisó no es un modelo: es una opinión con decimales.

---

### El problema

Trabajas en el equipo de percepción de una empresa de conducción autónoma. El vehículo lleva
un sensor **LiDAR** que, varias veces por segundo, detecta objetos alrededor y entrega para
cada uno una *caja delimitadora* (bounding box) con su posición, tamaño y velocidad estimada.

El equipo de modelamiento quiere entrenar un clasificador que distinga **peatones, ciclistas,
vehículos y señalética**. Antes de gastar una sola hora en eso, alguien tiene que responder:

> **¿Podemos confiar en estas detecciones? ¿Qué tan sucios están los datos y qué habría que
> arreglar antes de modelar?**

Ese alguien eres tú, hoy.

---

### Sobre los datos

El archivo `detecciones_waymo_like.csv` es un **dataset sintético** generado para esta clase.
Usa **el mismo esquema** del componente `lidar_box` del
[Waymo Open Dataset v2](https://waymo.com/open/), un conjunto de datos real de conducción
autónoma. Es sintético por dos razones honestas:

1. Los datos reales de Waymo pesan varios GB y su licencia no permite redistribuirlos.
2. Nos permite garantizar que los problemas de calidad que hay que descubrir **están ahí**.

Si quieres repetir este mismo análisis sobre datos **reales** de Waymo, el notebook
`00_opcional_waymo_real.ipynb` explica cómo hacerlo: el código de este notebook funciona igual,
porque el esquema es el mismo.

---

### Al final de la sesión debes entregar

Un **mini-informe en Markdown** (última celda del notebook) con:

- 5 hallazgos sobre la calidad de los datos, cada uno respaldado con una cifra;
- 3 decisiones de preprocesamiento, cada una con su justificación;
- 1 riesgo ético o de sesgo identificado en el dataset.

---
## Preparación del entorno

Ejecuta esta celda primero. Funciona tanto en Google Colab como en Jupyter local.

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO
else:
    # El notebook vive en notebooks/, así que la raíz del repositorio es la carpeta superior.
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"

print("Colab:", EN_COLAB)
print("Raíz del repositorio:", RAIZ)
print("¿Existe el dataset?:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import eda  # utilidades de diagnóstico del repositorio: src/eda.py

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

print("pandas", pd.__version__, "| numpy", np.__version__)

---
# Bloque 1 · Carga e inspección inicial

**Preguntas que debemos responder antes de tocar nada:**

1. ¿Cuántas filas y columnas hay? ¿Cuánta memoria ocupan?
2. ¿Qué representa **una fila**? (esta es la pregunta más importante y la que más se salta)
3. ¿El tipo de dato que pandas infirió para cada columna es el que corresponde?

In [ ]:
df = pd.read_csv(RUTA_DATOS)
print(f"Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}")
df.head()

### 📖 Diccionario de datos

| Columna | Significado |
|---|---|
| `segment_id` | Identificador del segmento de conducción (~20 s de grabación) |
| `timestamp_micros` | Instante de la detección, en microsegundos |
| `id_interno` | Identificador único de la detección |
| `object_type` | Tipo de objeto detectado |
| `box_center_x/y/z` | Centro de la caja, en metros, respecto del vehículo (x = adelante) |
| `box_length/width/height` | Dimensiones de la caja, en metros |
| `speed_mps` | Velocidad estimada del objeto, en m/s |
| `num_lidar_points` | Cantidad de puntos láser que cayeron sobre el objeto |
| `weather` | Condición climática del segmento |
| `time_of_day` | Momento del día |
| `detection_difficulty` | Dificultad de la detección según el sensor (LEVEL_1 = fácil) |
| `sensor_version` | Versión del firmware del sensor |

**Una fila = una detección de un objeto en un instante determinado.** No es un objeto, ni un
segmento, ni un vehículo. Ténlo presente: define qué significa "duplicado" más adelante.

### ✏️ TODO 1

Obtén, en una sola celda:

1. la estructura del DataFrame con `.info()`;
2. el uso de memoria **real** (`memory_usage(deep=True)`) en MB.

In [ ]:
# TODO 1: estructura del DataFrame y memoria real en MB
df.____()

memoria_mb = df.memory_usage(____).sum() / 1024**2
print(f"\nMemoria real: {memoria_mb:.1f} MB")

### ✏️ TODO 2 — El primer problema

Mira la salida anterior con atención. Hay una columna cuyo tipo **no es el que debería ser**.

1. Identifica qué columna es y por qué debería ser numérica.
2. Averigua qué valor la está ensuciando y cuántas filas lo tienen.

*Pista: `df["columna"].unique()` en una columna con miles de valores no sirve de mucho. Piensa
en qué le pasa a una columna numérica cuando aparece un texto.*

In [ ]:
# TODO 2: ¿qué valor ensucia la columna y cuántas filas lo tienen?
convertidos = pd.to_numeric(df["____"], errors="coerce")
no_convertibles = df.loc[convertidos.isna(), "____"]

print("dtype actual:", df["____"].dtype)
print("Valores no convertibles a número:", len(no_convertibles))
print(no_convertibles.value_counts())

In [ ]:
# Autochequeo
assert df["timestamp_micros"].dtype == object, "revisa: ¿estás mirando la columna correcta?"
assert len(no_convertibles) > 0, "deberías haber encontrado valores no convertibles"
print(f"✅ Hallazgo 1: {len(no_convertibles)} filas con un valor de texto en una columna numérica.")

### Diagnóstico general

En vez de revisar columna por columna a mano, usamos `eda.resumen_calidad()`, que entrega una
radiografía completa: tipo, cardinalidad, nulos y **valores centinela** (valores que
representan un dato faltante sin ser `NaN`, como `-1` o `"N/D"`).

In [ ]:
resumen = eda.resumen_calidad(df)
resumen

### ✏️ TODO 3

Usando la tabla anterior, responde en la celda de texto de abajo:

1. ¿Qué columna tiene **cardinalidad casi 100 %**? ¿Sirve como variable predictora? ¿Por qué?
2. ¿Qué columna es **constante**? ¿Qué aporta a un modelo?
3. ¿Qué columnas tienen nulos declarados (`NaN`) y cuáles tienen **nulos ocultos** (centinelas)?

In [ ]:
# TODO 3: filtra la tabla `resumen` para responder las tres preguntas.
# Pista: resumen[resumen["pct_unicos"] > 90], resumen[resumen["n_unicos"] <= 1], ...
print("Columnas de cardinalidad casi única (no son features):")
print(____)

print("\nColumnas constantes (no aportan información):")
print(____)

print("\nColumnas con algo faltante (declarado u oculto):")
print(____)

**✍️ Tu respuesta al TODO 3:**

*(doble clic aquí y escribe)*

1.
2.
3.

---
# Bloque 2 · Tipos de variables y categorías

El tipo que usa pandas (`int64`, `object`, …) no es lo mismo que el **tipo estadístico** de la
variable, y es el tipo estadístico el que decide qué se puede hacer con ella:

| Tipo estadístico | Definición | Ejemplo aquí | ¿Media? |
|---|---|---|---|
| **Nominal** | categorías sin orden | `object_type`, `weather` | ❌ |
| **Ordinal** | categorías con orden | `detection_difficulty` | ❌ (sí mediana) |
| **Discreta** | numérica, se cuenta | `num_lidar_points` | ✅ |
| **Continua** | numérica, se mide | `speed_mps`, `box_length` | ✅ |

`timestamp_micros` es un caso aparte: es numérica, pero su significado es **temporal**. Calcular
su promedio no tiene sentido; calcular diferencias, sí.

### ✏️ TODO 4

Completa el diccionario clasificando cada columna. Después ejecuta el autochequeo.

In [ ]:
# TODO 4: completa el tipo estadístico de cada columna.
# Opciones: nominal | ordinal | discreta | continua | temporal | identificador | constante
tipos_estadisticos = {
    "segment_id": "nominal",
    "timestamp_micros": "____",
    "id_interno": "____",
    "object_type": "____",
    "box_center_x": "continua",
    "box_center_y": "____",
    "box_center_z": "____",
    "box_length": "____",
    "box_width": "____",
    "box_height": "____",
    "speed_mps": "____",
    "num_lidar_points": "____",
    "weather": "____",
    "time_of_day": "____",
    "detection_difficulty": "____",
    "sensor_version": "____",
}

In [ ]:
# Autochequeo
faltantes = set(df.columns) - set(tipos_estadisticos)
assert not faltantes, f"faltan columnas por clasificar: {faltantes}"
assert "____" not in tipos_estadisticos.values(), "quedaron casilleros sin completar"
assert tipos_estadisticos["num_lidar_points"] == "discreta", "se cuentan puntos: es discreta"
assert tipos_estadisticos["detection_difficulty"] == "ordinal", "LEVEL_1 < LEVEL_2: hay orden"
print("✅ Clasificación completa y coherente.")

### El problema de las categorías

Ahora miremos qué categorías existen realmente en las variables nominales. Aquí es donde
aparecen los problemas que ningún `.info()` muestra.

### ✏️ TODO 5

Muestra la frecuencia de cada valor de `object_type` y de `weather`, **incluyendo los nulos**.

In [ ]:
# TODO 5: frecuencias incluyendo nulos (revisa el parámetro dropna)
print(df["object_type"].value_counts(____), "\n")
print(df["weather"].value_counts(____))

Cuenta las categorías que ves. ¿Cuántos tipos de objeto hay **en realidad**? ¿Cuántas condiciones
climáticas distintas existen **en realidad**?

### ✏️ TODO 6

Normaliza ambas columnas: quita espacios, unifica mayúsculas y traduce las variantes a una
forma canónica. Usa `eda.normalizar_categoria(serie, mapa)`.

*Ojo con `"RAIN "` y `" rain"`: los espacios son invisibles en pantalla pero `pandas` los cuenta
como categorías distintas.*

In [ ]:
# TODO 6: define los mapas de equivalencia y normaliza.
# El mapa se aplica DESPUÉS de pasar a minúsculas y quitar espacios: escribe las llaves así.
mapa_objetos = {"peaton": "pedestrian", "____": "pedestrian"}
mapa_clima = {"soleado": "sunny", "____": "rain", "____": "fog"}

df["object_type_limpio"] = eda.normalizar_categoria(df["object_type"], ____)
df["weather_limpio"] = eda.normalizar_categoria(df["weather"], ____)

print(df["object_type_limpio"].value_counts(dropna=False), "\n")
print(df["weather_limpio"].value_counts(dropna=False))

In [ ]:
# Autochequeo
assert set(df["object_type_limpio"].dropna().unique()) == {"vehicle", "pedestrian", "cyclist", "sign"}, \
    "deben quedar exactamente 4 tipos de objeto"
assert set(df["weather_limpio"].dropna().unique()) == {"sunny", "rain", "fog"}, \
    "deben quedar exactamente 3 condiciones climáticas"
print("✅ 7 variantes de objeto → 4 categorías | 11 variantes de clima → 3 categorías")

### Desbalance de clases

Con las categorías ya limpias, podemos ver algo que antes estaba oculto: cómo se reparten
las clases que el equipo quiere predecir.

In [ ]:
desbalance = eda.resumen_desbalance(df["object_type_limpio"])
print(desbalance)

fig, ax = plt.subplots(figsize=(7, 3.5))
desbalance["pct"].sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("% de detecciones")
ax.set_ylabel("")
ax.set_title("Composición del dataset por tipo de objeto")
for i, valor in enumerate(desbalance["pct"].sort_values()):
    ax.text(valor + 0.7, i, f"{valor:.1f}%", va="center")
plt.tight_layout()
plt.show()

---
# Bloque 3 · Datos faltantes y duplicados

Tres preguntas, en este orden:

1. ¿Cuántos faltan? (lo fácil)
2. ¿Están **escondidos** detrás de un valor válido? (lo que casi nadie revisa)
3. ¿Faltan **al azar** o siguen un patrón? (lo que decide qué podemos hacer con ellos)

### ✏️ TODO 7

Calcula, para cada columna, el número y el porcentaje de nulos declarados, mostrando solo las
columnas que tengan al menos uno.

In [ ]:
# TODO 7: conteo y porcentaje de nulos por columna, solo las que tengan alguno.
nulos = pd.DataFrame({
    "n_nulos": df.____().sum(),
    "pct": (100 * df.____().mean()).round(2),
})
nulos[nulos["n_nulos"] > 0].sort_values("n_nulos", ascending=False)

### Nulos ocultos

`isna()` solo ve lo que pandas reconoce como faltante. Un dato faltante también puede estar
disfrazado de valor válido: `-1`, `0`, `-999`, `"N/D"`, `"sin dato"`.

### ✏️ TODO 8

`num_lidar_points` es un conteo de puntos láser. Por definición **no puede ser negativo**.
Averigua cuántas filas violan esa regla y qué valor usan.

In [ ]:
# TODO 8: ¿cuál es el mínimo de num_lidar_points? ¿tiene sentido? ¿cuántas filas lo tienen?
print(df["num_lidar_points"].____(), "\n")
n_centinela = (df["num_lidar_points"] == ____).sum()
print(f"Filas con -1: {n_centinela:,} ({100 * n_centinela / len(df):.2f}%)")
print("Nulos que pandas ve en esa columna:", df["num_lidar_points"].isna().sum())

### ✏️ TODO 9

Crea las versiones corregidas de las dos columnas contaminadas, convirtiendo el valor centinela
en un `NaN` explícito:

- `num_lidar_points_limpio`: igual que la original, pero con `-1` → `NaN`.
- `timestamp_limpio`: la marca de tiempo convertida a número, con `"N/D"` → `NaN`.

In [ ]:
# TODO 9: convierte los valores centinela en NaN explícitos.
df["num_lidar_points_limpio"] = df["num_lidar_points"].replace(____, np.nan)
df["timestamp_limpio"] = eda.a_numerico(df["____"])

print(df[["num_lidar_points_limpio", "timestamp_limpio"]].isna().sum())
print("\nTipos:", df["num_lidar_points_limpio"].dtype, "|", df["timestamp_limpio"].dtype)

In [ ]:
# Autochequeo
assert df["num_lidar_points_limpio"].isna().sum() > 0, "los -1 deben quedar como NaN"
assert (df["num_lidar_points_limpio"].dropna() > 0).all(), "no pueden quedar conteos negativos"
assert pd.api.types.is_numeric_dtype(df["timestamp_limpio"]), "el timestamp debe ser numérico"
print("✅ Nulos ocultos convertidos en nulos explícitos.")

### ¿Los nulos son aleatorios?

Esta es **la** pregunta del bloque. Tres escenarios posibles:

| Mecanismo | Significa | Consecuencia |
|---|---|---|
| **MCAR** | falta al azar puro | eliminar filas es (casi) inofensivo |
| **MAR** | la falta depende de *otras* variables observadas | se puede imputar condicionando |
| **MNAR** | la falta depende del *propio* valor faltante | eliminar **sesga** el dataset |

Veamos el caso de `speed_mps`.

### ✏️ TODO 10

Cruza el porcentaje de nulos de `speed_mps` por `detection_difficulty` y `time_of_day`. Usa
`eda.matriz_nulos_por_grupo(df, columna, [grupo1, grupo2])`.

In [ ]:
# TODO 10: ¿el porcentaje de nulos es parejo entre grupos, o hay un patrón?
patron = eda.matriz_nulos_por_grupo(df, "____", ["____", "____"])
print(patron, "\n")

fig, ax = plt.subplots(figsize=(6.5, 3))
sns.heatmap(patron, annot=True, fmt=".1f", cmap="Reds", cbar_kws={"label": "% nulos"}, ax=ax)
ax.set_title("% de velocidad faltante según dificultad y momento del día")
plt.tight_layout()
plt.show()

**✍️ Discusión (escribe tu respuesta):**

Si el equipo decide `df.dropna(subset=["speed_mps"])` antes de entrenar:

1. ¿Qué tipo de detecciones desaparecen del dataset?
2. ¿Qué le pasa al modelo entrenado con lo que queda cuando el auto circula **de noche**?
3. ¿Es esto MCAR, MAR o MNAR?

*(doble clic y responde)*

### Duplicados

Recuerda: **una fila = una detección de un objeto en un instante**. Entonces, dos filas con el
mismo `id_interno` son, por definición, un error.

Hay dos tipos de duplicado y solo uno se resuelve con `drop_duplicates()`:

- **Duplicado exacto:** la fila completa está repetida.
- **Duplicado lógico:** se repite la *llave*, pero los demás valores difieren. `drop_duplicates()`
  no lo detecta, porque para pandas las filas son distintas.

### ✏️ TODO 11

Cuantifica ambos tipos usando `eda.reporte_duplicados(df, llave)` con la llave
`["segment_id", "timestamp_micros", "id_interno"]`, y muestra un ejemplo concreto de duplicado
lógico.

In [ ]:
# TODO 11: cuantifica duplicados exactos y lógicos, y muestra un ejemplo de duplicado lógico.
LLAVE = ["segment_id", "timestamp_micros", "id_interno"]
print(eda.reporte_duplicados(df, ____), "\n")

repetidas = df[df.duplicated(subset=LLAVE, keep=False)]
for _, grupo in repetidas.groupby(LLAVE):
    if grupo.drop_duplicates().shape[0] > 1:   # el grupo tiene filas distintas entre sí
        display(grupo[LLAVE + ["object_type", "box_center_x", "num_lidar_points"]])
        break

In [ ]:
# Autochequeo
reporte = eda.reporte_duplicados(df, LLAVE).iloc[0]
assert reporte["dup_exactos"] > 0, "hay duplicados exactos en el dataset"
assert reporte["dup_logicos"] > 0, "y también duplicados lógicos: drop_duplicates() no basta"
print(f"✅ {reporte['dup_exactos']} duplicados exactos + {reporte['dup_logicos']} duplicados lógicos.")

---
# Bloque 4 · Valores atípicos

Un valor atípico puede ser dos cosas muy distintas:

- un **error de medición** (el sensor falló) → hay que corregirlo o eliminarlo;
- un **caso real poco frecuente** (existe un bus) → eliminarlo es destruir información valiosa.

Los métodos estadísticos **no distinguen entre ambos**. Esa distinción la hace quien conoce el
dominio. Por eso este bloque no se trata de aplicar una fórmula, sino de mirar los datos.

In [ ]:
numericas = ["box_center_x", "box_center_y", "box_center_z",
             "box_length", "box_width", "box_height", "speed_mps"]
eda.perfil_numerico(df, numericas)

### ✏️ TODO 12

Mira la fila de `speed_mps` en la tabla anterior: compara la mediana (`50%`) con el máximo.

Grafica la distribución de `speed_mps` con un histograma y un boxplot, y responde: ¿es plausible
el máximo? (1 m/s = 3,6 km/h).

In [ ]:
# TODO 12: histograma y boxplot de speed_mps; luego convierte el máximo a km/h.
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
df["____"].plot.hist(bins=80, ax=axes[0], color="#4C72B0")
axes[0].set_title("Distribución de speed_mps")
axes[0].set_xlabel("m/s")

sns.boxplot(x=df["____"], ax=axes[1])
axes[1].set_title("Boxplot de speed_mps")
plt.tight_layout()
plt.show()

maximo = df["speed_mps"].____()
print(f"Máximo observado: {maximo:.1f} m/s = {maximo * 3.6:.0f} km/h")
print(f"Detecciones sobre 60 m/s (216 km/h): {(df['speed_mps'] > 60).sum()}")

### El criterio del rango intercuartil (IQR)

Se marca como atípico todo valor fuera del intervalo

$$[\,Q_1 - k\cdot IQR,\;\; Q_3 + k\cdot IQR\,], \qquad IQR = Q_3 - Q_1$$

con $k = 1{,}5$ para atípicos moderados y $k = 3$ para extremos.

Una alternativa es el **puntaje z**: $z = (x - \mu)/\sigma$, atípico si $|z| > 3$. Pero $\mu$ y
$\sigma$ se calculan *con* los outliers incluidos, así que un valor extremo infla $\sigma$ y se
esconde a sí mismo. El IQR, basado en cuantiles, es más robusto.

### ✏️ TODO 13

Compara ambos criterios sobre `speed_mps`: ¿cuántos valores marca cada uno?

In [ ]:
# TODO 13: compara el criterio IQR con el z-score sobre speed_mps.
por_iqr = eda.detectar_outliers_iqr(df["speed_mps"], k=____)
por_z = eda.detectar_outliers_zscore(df["speed_mps"], umbral=____)
inferior, superior = eda.limites_iqr(df["speed_mps"], k=1.5)

print(f"Límites IQR: [{inferior:.2f}, {superior:.2f}] m/s")
print(f"Marcados por IQR:     {por_iqr.sum():>5}")
print(f"Marcados por z-score: {por_z.sum():>5}")
print(f"Marcados por ambos:   {(por_iqr & por_z).sum():>5}")

### ✏️ TODO 14 — Atípico imposible vs. atípico legítimo

Ahora la parte que ninguna fórmula resuelve. Revisa los valores atípicos de `box_length`:

1. ¿Cuántos son **negativos**? ¿Puede existir un objeto de largo negativo?
2. Los objetos con largo mayor a 12 m, ¿son errores? Mira su ancho, su alto y su tipo antes de
   responder.

In [ ]:
# TODO 14: separa los atípicos imposibles de los legítimos.
atipicos_largo = eda.detectar_outliers_iqr(df["____"])
print(f"Atípicos de box_length según IQR: {atipicos_largo.sum()}\n")

print("--- Largo negativo (imposible) ---")
print(f"{(df['box_length'] < ____).sum()} filas\n")

print("--- Largo > 12 m: ¿error o realidad? ---")
grandes = df[df["box_length"] > 12]
print(grandes[["object_type_limpio", "box_length", "box_width", "box_height"]].describe().round(2))
print("\nTipos de objeto involucrados:", grandes["____"].unique())

**✍️ Tu conclusión:** ¿qué harías con cada uno de los dos grupos y por qué?

*(doble clic y responde)*

### ✏️ TODO 15 — Reglas de dominio

En lugar de confiar en un umbral estadístico, escribamos explícitamente **qué es imposible**
en este dominio. Completa el diccionario de reglas: cada valor es una expresión que describe
las filas **inválidas**.

In [ ]:
# TODO 15: completa las reglas de dominio (describen filas INVÁLIDAS).
reglas = {
    "largo no positivo": "box_length <= 0",
    "alto no positivo": "____",
    "ancho no positivo": "____",
    "velocidad sobre 60 m/s (216 km/h)": "speed_mps > ____",
    "conteo de puntos negativo": "____",
    "peatón más alto que 2.5 m": "object_type_limpio == 'pedestrian' and box_height > 2.5",
}
eda.valores_imposibles(df, reglas)

---
# Bloque 5 · De los hallazgos a las decisiones

Encontrar problemas es la mitad del trabajo. La otra mitad es **decidir qué hacer con cada uno
y dejarlo documentado**, porque cada decisión cambia los datos con los que se entrenará el
modelo.

### ✏️ TODO 16

Completa esta tabla con tus decisiones. Es el corazón de tu entrega.

| Columna | Problema detectado | Cifra | Decisión | Justificación |
|---|---|---|---|---|
| `timestamp_micros` | Valor `"N/D"` fuerza dtype texto | 60 filas | Convertir a numérico, `"N/D"` → `NaN` | Se preserva la fila; solo se pierde el instante |
| `num_lidar_points` | `-1` como nulo oculto | | | |
| `weather` | | | | |
| `object_type` | | | | |
| duplicados | | | | |
| `box_length` | | | | |
| `speed_mps` | | | | |
| `sensor_version` | | | | |

*(doble clic para editar la tabla)*

### ✏️ TODO 17 — Aplica tus decisiones

Escribe una función que reciba el DataFrame crudo y devuelva el limpio. Que sea una función y no
celdas sueltas importa: es reproducible, se puede testear y se puede volver a aplicar a datos
nuevos.

In [ ]:
# TODO 17: completa la función de limpieza con TUS decisiones del TODO 16.
def limpiar(datos: pd.DataFrame) -> pd.DataFrame:
    """Aplica las decisiones de preprocesamiento acordadas y devuelve una copia limpia."""
    d = datos.copy()

    # 1. Tipos y valores centinela
    d["timestamp_micros"] = eda.a_numerico(d["timestamp_micros"])
    d["num_lidar_points"] = d["num_lidar_points"].replace(____, np.nan)

    # 2. Categorías
    d["object_type"] = eda.normalizar_categoria(d["object_type"], ____)
    d["weather"] = eda.normalizar_categoria(d["weather"], ____)
    d["weather"] = d["weather"].fillna("desconocido")

    # 3. Valores físicamente imposibles -> faltantes
    for columna in ["box_length", "box_width", "box_height"]:
        d.loc[d[columna] <= 0, columna] = np.nan
    d.loc[d["speed_mps"] > ____, "speed_mps"] = np.nan

    # 4. Indicador de faltante
    d["speed_faltante"] = d["speed_mps"].isna().astype(int)

    # 5. Duplicados: exactos y luego lógicos, con una regla de desempate explícita
    d = d.____()
    d = (d.sort_values("num_lidar_points", ascending=False, na_position="last")
           .drop_duplicates(subset=["segment_id", "timestamp_micros", "id_interno"], keep="first")
           .sort_index())

    # 6. Columnas sin valor predictivo
    d = d.drop(columns=[____])

    return d


df_limpio = limpiar(pd.read_csv(RUTA_DATOS))
print(f"Crudo:  {len(df):,} filas")
print(f"Limpio: {len(df_limpio):,} filas  ({len(df) - len(df_limpio):,} eliminadas)")
df_limpio.head(3)

In [ ]:
# Autochequeo del dataset limpio
assert df_limpio.duplicated(subset=["segment_id", "timestamp_micros", "id_interno"]).sum() == 0, \
    "no deben quedar llaves repetidas"
assert set(df_limpio["object_type"].unique()) == {"vehicle", "pedestrian", "cyclist", "sign"}
assert "sensor_version" not in df_limpio.columns
assert (df_limpio["box_length"].dropna() > 0).all(), "no deben quedar largos imposibles"
assert df_limpio["box_length"].max() > 12, "los buses deben SEGUIR AHÍ: no son errores"
assert pd.api.types.is_numeric_dtype(df_limpio["timestamp_micros"])
print("✅ Dataset limpio y auditable. Los casos raros pero reales siguen presentes.")

### ⚠️ Una advertencia para las próximas semanas: la fuga de información

Fíjate en algo que **no** hicimos: no imputamos los nulos con la media ni escalamos ninguna
variable.

No es un olvido. Si calculas la media de todo el dataset y con ella rellenas los nulos, y
**después** separas entrenamiento y prueba, el conjunto de prueba ya influyó en el conjunto de
entrenamiento a través de esa media. Eso se llama **fuga de información** (*data leakage*), y su
síntoma es un modelo que rinde excelente en las pruebas y mal en producción.

El orden correcto es:

```
limpieza estructural (lo de hoy)  →  separar train/test  →  ajustar imputación y escalado SOLO con train  →  aplicar a test
```

En EA2 haremos esto con `Pipeline` y `ColumnTransformer` de scikit-learn, que existen justamente
para que este error sea difícil de cometer.

---
# Bloque 6 · Tratamiento responsable de la información

Los datos de conducción autónoma se recogen **en la vía pública**, donde hay personas que nunca
dieron su consentimiento. Antes de modelar, tres preguntas:

1. **¿Hay datos personales aquí?** El dataset no tiene nombres ni rostros, pero sí posiciones de
   peatones asociadas a un instante y un segmento. Si el segmento tiene geolocalización (los
   datos reales de Waymo la tienen), la combinación *lugar + hora + trayectoria* puede
   reidentificar a una persona. Anonimizar no es solo borrar la columna "nombre".
2. **¿Cómo se recolectó?** En los datos reales, con cámaras y LiDAR en vía pública. Waymo difumina
   rostros y patentes antes de publicar. Esa decisión es parte del diseño del dataset, no un
   detalle técnico.
3. **¿A quién representa mal este dataset?** Es la pregunta del ejercicio siguiente.

### ✏️ TODO 18

Calcula la composición del dataset por clima y momento del día, y el porcentaje de detecciones
que ocurren de noche o con lluvia.

In [ ]:
# TODO 18: composición por clima y hora; luego los porcentajes de noche, lluvia y ciclistas.
composicion = pd.crosstab(df_limpio["____"], df_limpio["____"], normalize="all") * 100
print(composicion.round(2), "\n")

pct_noche = 100 * (df_limpio["time_of_day"] == "____").mean()
pct_lluvia = 100 * (df_limpio["weather"] == "____").mean()
pct_dificil_noche = 100 * ((df_limpio["time_of_day"] == "Night") &
                           (df_limpio["detection_difficulty"] == "LEVEL_2")).mean()

print(f"Detecciones nocturnas: {pct_noche:.1f}%")
print(f"Detecciones con lluvia: {pct_lluvia:.1f}%")
print(f"Nocturnas Y difíciles: {pct_dificil_noche:.1f}%")
print(f"Ciclistas: {100 * (df_limpio['object_type'] == 'cyclist').mean():.1f}%")

**✍️ Discusión final (escribe tu respuesta):**

Un modelo entrenado con este dataset se instalará en autos que circulan **de noche y con lluvia**,
y que se cruzan con **ciclistas**.

1. ¿Qué situación está sub-representada en los datos?
2. ¿Qué grupo de personas corre más riesgo si el modelo falla en esa situación?
3. Nombra **una** medida concreta que propondrías antes de desplegar este modelo.

*(doble clic y responde)*

---
# 📝 Entrega: mini-informe

Completa la celda siguiente. Es lo que entregas al final de la sesión.

**Reglas:**
- Cada hallazgo debe incluir **una cifra**. "Hay datos sucios" no es un hallazgo; "el 3 % de
  `num_lidar_points` usa `-1` como nulo oculto" sí lo es.
- Cada decisión debe incluir **una justificación**, no solo qué hiciste.

## Informe de calidad de datos — EA1

**Estudiante:**
**Fecha:**

### Contexto
*(¿qué problema se quiere resolver y qué representa una fila del dataset?)*

### 5 hallazgos

| # | Hallazgo | Cifra | Impacto en el modelo |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

### 3 decisiones de preprocesamiento

| # | Decisión | Justificación | Qué se pierde |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

### 1 riesgo ético o de sesgo


### Conclusión
*(en 3 líneas: ¿está este dataset listo para entrenar un modelo? ¿qué falta?)*

---
### ✅ Antes de cerrar

- [ ] Todas las celdas ejecutan sin error (Kernel → Restart & Run All).
- [ ] Los autochequeos muestran ✅.
- [ ] El mini-informe está completo, con cifras.
- [ ] Las celdas de discusión tienen tu respuesta escrita.

### Lo que viene

- **Semana 2:** técnicas de preprocesamiento aplicadas (imputación, codificación, escalado) y
  cómo evitar la fuga de información con `Pipeline`.
- **EA2:** aprendizaje supervisado — regresión y clasificación. Ahí veremos por qué ese 2 % de
  ciclistas es un problema serio.
- **EA3:** aprendizaje no supervisado — segmentación y reducción de dimensionalidad.

### ¿Quieres hacerlo con datos reales?

El notebook `00_opcional_waymo_real.ipynb` explica cómo bajar un fragmento del Waymo Open
Dataset real y correr **este mismo análisis** sobre él. El esquema es el mismo; el código, casi
idéntico.